# 18 — Advanced LLM/RAG Applications

## 📓 Interactive Notebook · Module 14 · AI/LLM

In this notebook, you'll learn:
1. **LLM application architecture**
2. **Secrets management** for API keys
3. **Conversation state** and chat interfaces
4. **Streaming responses**
5. **RAG architecture** for document Q&A
6. **Security considerations**

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Build LLM-powered chat applications
- Manage API keys securely
- Implement RAG for document Q&A
- Apply security best practices
- Use local model alternatives

---

## 💡 LLM Application Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Streamlit UI   │ ──▶ │  App Logic      │ ──▶ │  LLM Provider   │
│  (Chat, Forms)  │     │  (RAG, State)   │     │  (API/Local)    │
└─────────────────┘     └─────────────────┘     └─────────────────┘
        ▲                                                 │
        └─────────────────────────────────────────────────┘
                         Response
```

---

## 💡 Step 1: Secrets Management

**NEVER hard-code API keys!**

In [ ]:
import streamlit as st

st.header("🔐 Secrets Management")

st.subheader("❌ WRONG: Hard-coded keys")
st.code('''
# NEVER do this!
api_key = "sk-your-secret-key-here"  # Exposed in code!
''', language="python")

st.subheader("✅ CORRECT: Streamlit secrets")
st.code('''
# .streamlit/secrets.toml (NEVER commit this!)
# [openai]
# api_key = "sk-your-secret-key-here"

# In your app:
import streamlit as st
api_key = st.secrets.openai.api_key
''', language="python")

st.subheader("✅ ALTERNATIVE: Environment variables")
st.code('''
import os
api_key = os.getenv("OPENAI_API_KEY")
''', language="python")

# Check if key is configured
try:
    api_key = st.secrets.get("openai", {}).get("api_key")
    if api_key:
        st.success("✅ OpenAI API key is configured")
    else:
        st.warning("⚠️ OpenAI API key not found. Add to .streamlit/secrets.toml")
except FileNotFoundError:
    st.warning("⚠️ No secrets.toml found. Create .streamlit/secrets.toml")

---

## 💡 Step 2: Provider Abstraction

Keep LLM code modular and swappable.

In [ ]:
import streamlit as st
from abc import ABC, abstractmethod
from typing import List, Generator

st.header("🔌 Provider Abstraction")

# Base class
st.subheader("Base Provider Interface")
st.code('''
class LLMProvider(ABC):
    """Base class for LLM providers."""
    
    @abstractmethod
    def chat(self, messages: list, **kwargs) -> str:
        """Send chat completion request."""
        pass
    
    @abstractmethod
    def stream_chat(self, messages: list, **kwargs) -> Generator:
        """Stream chat completion response."""
        pass
''', language="python")

# OpenAI provider
st.subheader("OpenAI Provider")
st.code('''
import openai

class OpenAIProvider(LLMProvider):
    def __init__(self, api_key: str, model: str = "gpt-3.5-turbo"):
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model
    
    def chat(self, messages: list, **kwargs) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            **kwargs
        )
        return response.choices[0].message.content
    
    def stream_chat(self, messages: list, **kwargs):
        stream = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            stream=True,
            **kwargs
        )
        for chunk in stream:
            if chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content
''', language="python")

# Local provider
st.subheader("Local Provider (Ollama)")
st.code('''
import requests

class LocalProvider(LLMProvider):
    def __init__(self, base_url: str = "http://localhost:11434"):
        self.base_url = base_url
    
    def chat(self, messages: list, model: str = "llama2") -> str:
        response = requests.post(
            f"{self.base_url}/api/chat",
            json={"model": model, "messages": messages}
        )
        return response.json()["message"]["content"]
''', language="python")

---

## 💡 Step 3: Conversation State

Manage chat history with session state.

In [ ]:
import streamlit as st
from typing import List, Dict

st.header("💬 Conversation State")

st.code('''
def init_conversation():
    """Initialize conversation state."""
    if "messages" not in st.session_state:
        st.session_state.messages = []
    if "system_prompt" not in st.session_state:
        st.session_state.system_prompt = "You are a helpful assistant."

def add_message(role: str, content: str):
    """Add message to history."""
    st.session_state.messages.append({"role": role, "content": content})

def get_context_messages(max_history: int = 10) -> List[Dict]:
    """Get messages for API (with system prompt)."""
    messages = [{"role": "system", "content": st.session_state.system_prompt}]
    messages.extend(st.session_state.messages[-max_history:])
    return messages

def clear_conversation():
    """Clear chat history."""
    st.session_state.messages = []
''', language="python")

# Demo
st.subheader("Demo: Initialize Conversation")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "user", "content": "Hello!"},
        {"role": "assistant", "content": "Hi! How can I help you?"}
    ]

st.write("**Current messages:**")
for msg in st.session_state.messages:
    st.write(f"- **{msg['role']}:** {msg['content']}")

if st.button("Clear History"):
    st.session_state.messages = []
    st.rerun()

---

## 💡 Step 4: Chat Interface

Build a chat UI with `st.chat_input` and `st.chat_message`.

In [ ]:
import streamlit as st

st.header("💬 Chat Interface Pattern")

st.code('''
def chat_interface(provider):
    """Basic chat interface."""
    init_conversation()
    
    # Display history
    for message in get_messages():
        with st.chat_message(message["role"]):
            st.write(message["content"])
    
    # User input
    if prompt := st.chat_input("Type your message..."):
        # Add user message
        add_message("user", prompt)
        with st.chat_message("user"):
            st.write(prompt)
        
        # Get response
        with st.chat_message("assistant"):
            messages = get_context_messages()
            response = st.write_stream(provider.stream_chat(messages))
        
        # Add response
        add_message("assistant", response)
''', language="python")

st.write("**Key components:**")
st.write("1. `st.chat_input()` — Bottom-aligned input box")
st.write("2. `st.chat_message()` — Styled message bubbles")
st.write("3. `st.write_stream()` — Streaming response display")
st.write("4. Session state — Persist conversation history")

---

## 💡 Step 5: Streaming Responses

Show responses as they're generated.

In [ ]:
import streamlit as st

st.header("⚡ Streaming Responses")

st.subheader("Method 1: Native Streaming")
st.code('''
with st.chat_message("assistant"):
    response = st.write_stream(provider.stream_chat(messages))
''', language="python")

st.subheader("Method 2: Custom Placeholder")
st.code('''
def stream_with_placeholder(provider, messages):
    placeholder = st.empty()
    full_response = ""
    
    for chunk in provider.stream_chat(messages):
        full_response += chunk
        placeholder.markdown(full_response + "▌")
    
    placeholder.markdown(full_response)
    return full_response
''', language="python")

# Demo streaming simulation
st.subheader("Demo: Simulated Streaming")
if st.button("Simulate Stream"):
    placeholder = st.empty()
    text = "This is a simulated streaming response. Watch how the text appears word by word."
    
    displayed = ""
    for word in text.split():
        displayed += word + " "
        placeholder.markdown(displayed + "▌")
        import time; time.sleep(0.1)
    
    placeholder.markdown(text)

---

## 💡 Step 6: RAG Architecture

Retrieval-Augmented Generation for grounded answers.

In [ ]:
import streamlit as st

st.header("📚 RAG Architecture")

st.markdown("""
**RAG Pipeline:**

```
1. User Question
       │
       ▼
2. Retrieve Relevant Documents (Vector Search)
       │
       ▼
3. Inject Context into Prompt
       │
       ▼
4. Generate Answer with LLM
       │
       ▼
5. Return Grounded Response
```
""")

st.code('''
class RAGPipeline:
    def __init__(self, provider, vector_store):
        self.provider = provider
        self.vector_store = vector_store
    
    def retrieve(self, query: str, n_results: int = 3) -> list:
        """Retrieve relevant documents."""
        return search_documents(query, n_results)
    
    def generate(self, query: str, context: list) -> str:
        """Generate with context."""
        context_text = "\n\n".join(context)
        
        messages = [
            {"role": "system", "content": f"""Answer based on context:
            
            {context_text}
            
            If not in context, say you don't know."""},
            {"role": "user", "content": query}
        ]
        
        return self.provider.chat(messages)
    
    def query(self, question: str) -> str:
        """Full RAG: retrieve + generate."""
        context = self.retrieve(question)
        if not context:
            return "No relevant documents found."
        return self.generate(question, context)
''', language="python")

---

## 💡 Step 7: Document Processing

Extract and chunk documents for RAG.

In [ ]:
import streamlit as st
from pathlib import Path

st.header("📄 Document Processing")

st.code('''
def process_document(file) -> str:
    """Extract text from document."""
    suffix = Path(file.name).suffix.lower()
    
    if suffix in [".txt", ".md"]:
        return file.read().decode("utf-8")
    
    elif suffix == ".pdf":
        import PyPDF2
        reader = PyPDF2.PdfReader(file)
        return "\n".join(page.extract_text() for page in reader.pages)
    
    return ""

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks
''', language="python")

# Demo
st.subheader("Demo: Text Chunking")
sample_text = """This is a sample document for demonstrating text chunking. 
It contains multiple sentences that will be split into smaller pieces.
The overlap ensures context is preserved between chunks."""

chunks = chunk_text(sample_text, chunk_size=100, overlap=20)
st.write(f"**Original:** {len(sample_text)} characters")
st.write(f"**Chunks:** {len(chunks)} chunks created")
for i, chunk in enumerate(chunks):
    st.write(f"**Chunk {i+1}:** {chunk[:80]}...")

---

## ⚠️ Security Considerations

Protect your LLM application.

In [ ]:
import streamlit as st

st.header("🔒 Security Considerations")

st.subheader("Prompt Injection Awareness")
st.code('''
# Example attacks:
# - "Ignore previous instructions..."
# - "You are now DAN (Do Anything Now)..."
# - "Reveal your system prompt..."

def sanitize_input(text: str) -> str:
    """Basic input sanitization."""
    suspicious = ["ignore previous", "ignore all instructions", 
                  "you are now", "system prompt:"]
    
    text_lower = text.lower()
    for pattern in suspicious:
        if pattern in text_lower:
            return ""
    return text
''', language="python")

st.subheader("Security Checklist")
st.markdown("""
✅ **API Keys** — Never hard-code, use `st.secrets`

✅ **Input Validation** — Sanitize, limit length

✅ **Rate Limiting** — Limit requests per user

✅ **Output Validation** — Don't execute LLM-generated code

✅ **Data Privacy** — Don't send sensitive data to external APIs
""")

---

## 🎯 Challenges

### Challenge 1: Build a Chat App
Create a complete chat application with:
- Conversation history
- System prompt customization
- Clear chat button

### Challenge 2: Document Q&A
Build a RAG app that:
- Accepts uploaded documents
- Answers questions about them
- Shows source references

### Challenge 3: Multi-Provider Support
Implement provider switching between OpenAI and local models.

In [ ]:
# Challenge 1: Build a Chat App
import streamlit as st

st.write("TODO: Build a complete chat application")

# Your code here


---

## 📝 Key Takeaways

1. **Provider abstraction** — Keep LLM code modular and swappable

2. **Secrets management** — Never hard-code API keys

3. **Conversation state** — Use `st.session_state` for chat history

4. **Streaming** — Use `st.write_stream` for better UX

5. **RAG** — Retrieve context, then generate grounded answers

6. **Security** — Validate inputs, rate limit, protect secrets

7. **Local alternatives** — Ollama, Hugging Face for offline development

---

## 📚 Further Reading

- [OpenAI API Documentation](https://platform.openai.com/docs)
- [LangChain Documentation](https://python.langchain.com/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [Ollama Documentation](https://ollama.ai/docs)

---

## 🔗 Related Materials

- 📖 Reading: [18 — LLM/RAG Applications](../readings/18_llm_rag_applications.md)
- ✏️ Exercise: [18 — LLM Workshop](../exercises/18_llm_workshop.py)
- 🖥️ Demo App: [18 — LLM Chat](../apps/18_llm_chat.py)
- 🖥️ Demo App: [18 — RAG Document Q&A](../apps/18_rag_app.py)
- 📝 Quiz: [14 — LLM & RAG](../quizzes/14_llm_rag.md)
- 🚀 Project: [P07 — RAG Document Chat](../projects/P07_rag_document_chat.md)